In [2]:
from app.chains.chat_pipeline_mcp import chat_once
s = await chat_once("What is postpartum care?", "6936df2b17d183e968c376ad")
s

llm_response-------> IN_SCOPE
llm_response-2232333------> True
[CHAT] Fetching context for user_id: 6936df2b17d183e968c376ad
[CHAT] MCP server script: /Users/rahul39duttagmail.com/nexaneura/rag_chatbot/app/mcp/context_server.py


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[CHAT] ✅ Profile context loaded
[CHAT] ✅ Recovery context loaded
[CHAT] Final prompt constructed. You are a compassionate postpartum **wellness** assistant for educational purposes only.
- You do NOT diagnose, prescribe, or provide medication dosages.
- Prefer using provided CONTEXT for facts; if CONTEXT is empty, answer from general postpartum wellness knowledge.
- For product questions, use PRODUCT_CANDIDATES if present; never invent products or claims.
- Keep responses concise, empathetic, practical, and culturally aware for users in India.
- If unsure or outside wellness scope, say so and suggest consulting a clinician.
- Always end with: 'Wellness information only; not medical advice.'
=== USER PROFILE ===
You are assisting Call me harsha, who is in postpartum week 2, day 1 (15 days after delivery). Delivery type: c_section. The delivery resulted in a live birth. Social support: family_help. Pregnancy conditions: anemia, gestational_diabetes, high_bp, obesity, thyroid, fibroids, t

{'session_id': '25992a2d-93c1-4976-be73-a1a8ad4f9fe3',
 'answer': "Harsha, I can sense that you're taking the first steps towards prioritizing your well-being after the birth of your baby, and that's truly commendable. It's completely normal to feel overwhelmed, especially considering your previous health conditions and the recent c-section delivery. I'm here to support you with empathy and understanding.\n\nPostpartum care refers to the support and attention you receive after giving birth, focusing on your physical, emotional, and mental recovery. Given your current recovery status, with a trend of improvement, it's essential to continue prioritizing your well-being. Your recent scores indicate that you're in the yellow zone for physical and lactation aspects, and the red zone for emotional aspects, which suggests that you might need some extra support and care in managing your emotions.\n\nTo take care of yourself, consider the following steps:\n\n1. **Reach out to your family suppor

In [ ]:
"""
Test script for recommendations_tool

Tests fetching the last 3 recommendations and formatting for the chatbot.

Run: python3 tests/test_recommendations.py
"""

import sys
import os

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(".env"))))

from dotenv import load_dotenv
load_dotenv()

from app.mcp.tools.get_active_recommendations_tool import (
    get_active_recommendations,
    format_recommendations_for_prompt,
    print_recommendations_summary
)
from app.mcp.db_connection import get_recommendation_history_collection

from app.mcp.db_connection import get_users_collection


def test_user_profile_tool():
    """Test the user profile fetching functionality"""
        

def test_recommendations_tool():
    """Test the recommendations fetching functionality"""
    print("\n" + "="*60)
    print("Testing Recommendations Tool")
    print("="*60 + "\n")
    print("\n" + "="*60)
    print("Testing User Profile Tool")
    print("="*60 + "\n")
        # First, let's find a real user in the database to test with
    users = get_users_collection()
    print(users, "223")
    sample_user = users.find_one({})
    print(sample_user, "user sample")
    try:
        
        # Find a user who has recommendation history
        rec_history = get_recommendation_history_collection()
        print(rec_history, "2232")
        sample_rec = rec_history.find_one({})
        print(sample_rec, "3232")
        
        if not sample_rec:
            print("ℹ️  No recommendation history in database yet.")
            print("   Users need to complete weekly check-ins first.")
            return False
        
        # Get the userId from the sample
        user_id = sample_rec.get("userId")
        if not user_id:
            print("❌ Sample recommendation has no userId field")
            return False
        
        user_id_str = str(user_id)
        
        print(f"✅ Found user with recommendation history")
        print(f"   User ID: {user_id_str}")
        print()
        
        # Test 1: Fetch recommendations
        print("-" * 60)
        print("Test 1: Fetching Last 3 Recommendations")
        print("-" * 60 + "\n")
        
        recs_data = get_active_recommendations(user_id_str, limit=3)
        
        if recs_data.get("found"):
            print(f"✅ Successfully fetched recommendations!")
            print(f"   Count: {recs_data['count']}")
            print(f"   Trend: {recs_data['trend']}")
            print()
            
            # Show each recommendation briefly
            for i, rec in enumerate(recs_data["recommendations"]):
                marker = "← CURRENT" if i == len(recs_data["recommendations"]) - 1 else ""
                print(f"   Week {rec['week']}: Score {rec['finalScore']}, {rec['zone']} zone {marker}")
        else:
            print(f"❌ Failed to fetch: {recs_data.get('error')}")
            return False
        
        # Test 2: Format for LLM
        print("\n" + "-" * 60)
        print("Test 2: Formatting for LLM Prompt")
        print("-" * 60 + "\n")
        
        formatted = format_recommendations_for_prompt(recs_data)
        print("✅ Formatted context:")
        print()
        print(formatted)
        print()
        
        # Test 3: Verify latest recommendation has content
        print("-" * 60)
        print("Test 3: Checking Recommendation Content")
        print("-" * 60 + "\n")
        
        latest = recs_data.get("latest")
        if latest and latest.get("content"):
            content = latest["content"]
            print("✅ Latest recommendation has content:")
            print(f"   Title: {content.get('title', 'N/A')[:60]}...")
            print(f"   Going Well: {content.get('goingWell', 'N/A')[:60]}...")
            print(f"   Focus: {content.get('needsHelp', 'N/A')[:60]}...")
        else:
            print("⚠️  Latest recommendation missing content")
            print("   (This might be normal if recommendations haven't been fully set up)")
        
        # Test 4: Pretty print
        print("\n" + "-" * 60)
        print("Test 4: Pretty Print Summary")
        print("-" * 60)
        
        print_recommendations_summary(user_id_str)
        
        return True
        
    except Exception as e:
        print(f"❌ Error during testing: {str(e)}")
        import traceback
        traceback.print_exc()
        return False


def test_with_specific_user():
    """Test with a specific user ID"""
    print("\n" + "="*60)
    print("Test with Specific User ID")
    print("="*60 + "\n")
    
    user_id = input("Enter a user_id to test (or press Enter to skip): ").strip()
    
    if not user_id:
        print("Skipped.")
        return
    
    print()
    print_recommendations_summary(user_id)


def main():
    """Run all tests"""
    print("\n🏥 Viva Mama - Recommendations Tool Tests")
    print("="*60)
    
    success = test_recommendations_tool()
    
    print("\n" + "="*60)
    if success:
        print("🎉 All tests passed!")
        print("="*60 + "\n")
        print("The recommendations tool is working correctly.")
        print()
        print("What this means:")
        print("✅ Can fetch last 3 recommendations")
        print("✅ Can calculate recovery trend (improving/stable/declining)")
        print("✅ Can link to full recommendation content")
        print("✅ Can format for chatbot consumption")
        print()
        print("This allows the chatbot to:")
        print("  - Reference current week's focus")
        print("  - Acknowledge progress from previous weeks")
        print("  - Celebrate improvements")
        print("  - Address ongoing concerns")
        print()
        
        test_with_specific_user()
    else:
        print("⚠️  Some tests failed")
        print("="*60 + "\n")
    
    return 0 if success else 1


if __name__ == "__main__":
    exit(main())